# New columns, apply and pd.cut

**Video 6.4** | ShopKart project

**Goal:** Add revenue, profit and order_month to the cleaned orders, label orders with apply, and group customer ages with pd.cut.

### How to open in Colab
1. Go to colab.research.google.com → **File → Upload notebook** → choose this `.ipynb` file.
2. Click the **folder icon** on the left sidebar (Files panel) and drag in the data files: `orders_clean.csv`, `products.csv`, `customers.csv`.
3. Run cells one by one with **Shift + Enter**.

> Files you upload to the Files panel disappear when the runtime resets. If you see `FileNotFoundError`, upload them again.

In [1]:
import pandas as pd
df = pd.read_csv("orders_clean.csv", parse_dates=["order_date"])   # parse_dates keeps dates as dates
products = pd.read_csv("products.csv")
customers = pd.read_csv("customers.csv")
print(df.shape, df["order_date"].dtype)

(5000, 12) datetime64[us]


## 1. Revenue - one line for 5,000 rows
Same formula as the Excel column in Module 2: quantity x unit_price x (1 - discount).

In [2]:
df["revenue"] = (df["quantity"] * df["unit_price"] * (1 - df["discount"])).round(2)
df[["order_id", "quantity", "unit_price", "discount", "revenue"]].head()

,order_id,quantity,unit_price,discount,revenue
0,O10001,6,362,0.20,1737.6
1,O10002,2,300,0.05,570.0
2,O10003,1,2258,0.05,2145.1
3,O10004,2,2020,0.15,3434.0
4,O10005,2,1937,0.10,3486.6


In [3]:
print("Total revenue 2025: ₹", round(df["revenue"].sum(), 2))
print("Average order value: ₹", round(df["revenue"].mean(), 2))

Total revenue 2025: ₹ 14704649.7
Average order value: ₹ 2940.93


## 2. Profit - needs the cost price from products
Cost price lives in `products.csv`. For now we look it up with `map` (like XLOOKUP). In video 6.6 we will do this properly with `merge`.

In [4]:
cost_lookup = products.set_index("product_id")["cost_price"]   # product_id -> cost_price
unit_cost = df["product_id"].map(cost_lookup)
df["profit"] = (df["revenue"] - df["quantity"] * unit_cost).round(2)
print("Total profit: ₹", round(df["profit"].sum(), 2))
print("Loss-making orders:", (df["profit"] < 0).sum())

Total profit: ₹ 3140749.7
Loss-making orders: 241


## 3. Date parts with .dt

In [5]:
df["order_month"] = df["order_date"].dt.month        # 1 to 12
df["month_name"] = df["order_date"].dt.month_name()  # 'January'...
df["weekday"] = df["order_date"].dt.day_name()
df[["order_date", "order_month", "month_name", "weekday"]].head()

,order_date,order_month,month_name,weekday
0,2025-01-01,1,January,Wednesday
1,2025-01-01,1,January,Wednesday
2,2025-01-01,1,January,Wednesday
3,2025-01-01,1,January,Wednesday
4,2025-01-01,1,January,Wednesday


## 4. apply: run your own function on every row

In [6]:
def order_size(revenue):
    if revenue >= 5000:
        return "Big"
    elif revenue >= 1000:
        return "Medium"
    return "Small"

df["order_size"] = df["revenue"].apply(order_size)
df["order_size"].value_counts()

order_size
Medium    2802
Small     1638
Big        560
Name: count, dtype: int64

In [7]:
# A simple yes/no column does not need apply - a comparison is faster
df["is_returned"] = (df["returned"] == "Y").astype(int)   # 1 = returned, 0 = kept
print("Return rate:", round(df["is_returned"].mean() * 100, 1), "%")

Return rate: 13.7 %


## 5. pd.cut: turn ages into groups
Ages run from 18 to 57. `bins=[17, 25, 35, 45, 60]` means (17,25], (25,35], ... - the right edge is included.

In [8]:
print(customers["age"].min(), customers["age"].max())
customers["age_group"] = pd.cut(customers["age"], bins=[17, 25, 35, 45, 60],
                                 labels=["18-25", "26-35", "36-45", "46-60"])
customers["age_group"].value_counts().sort_index()

18 57


age_group
18-25    233
26-35    395
36-45    151
46-60     21
Name: count, dtype: int64

In [9]:
customers[["customer_id", "name", "age", "age_group"]].head()

,customer_id,name,age,age_group
0,C0001,Neha Kapoor,18,18-25
1,C0002,Sneha Das,36,36-45
2,C0003,Rohit Kulkarni,20,18-25
3,C0004,Nisha Khan,26,26-35
4,C0005,Priya Khan,27,26-35


## Practice
Create a column `delivery_speed`: "Fast" for 2-3 days, "Normal" for 4-5, "Slow" for 6+ using `pd.cut` with bins `[0, 3, 5, 20]`. Count each.

In [10]:
df["delivery_speed"] = pd.cut(df["delivery_days"], bins=[0, 3, 5, 20], labels=["Fast", "Normal", "Slow"])
df["delivery_speed"].value_counts()

delivery_speed
Normal    2152
Fast      1905
Slow       943
Name: count, dtype: int64